# Блок 11 — asyncio: эксперименты

Перед каждым экспериментом — **Прогноз** (до запуска), после — **Вывод** (своими словами).
В ноутбуке loop уже запущен → `await` прямо в ячейке, без `asyncio.run`.

In [1]:
import asyncio
import time

import async_tasks
from async_tasks import STATS, fetch_all, fetch_one_by_one, reset_stats

## Эксперимент 1. `for … await` против `fetch_all`

10 id, все delay 0.3; замеряем `fetch_one_by_one` и `fetch_all` (3 прогона).

### Прогноз

**1. Сколько примерно секунд займёт каждый вариант?** – fetch_one_by_one – 3 сек, fetch_all – 0.3

**2. Чему будет равен `STATS["max_active"]` в каждом варианте?** – fetch_one_by_one – 1, в fetch_all – 10

In [2]:
ids = list(range(10))
delays = [0.3] * 10

for run in range(1, 4):
    for name, fn in [("one_by_one", fetch_one_by_one), ("fetch_all", fetch_all)]:
        reset_stats()
        t0 = time.perf_counter()
        await fn(ids, delays)
        print(f"прогон {run} | {name:10} | {time.perf_counter() - t0:.2f} с | max_active = {STATS['max_active']}")

прогон 1 | one_by_one | 3.01 с | max_active = 1


прогон 1 | fetch_all  | 0.30 с | max_active = 10


прогон 2 | one_by_one | 3.01 с | max_active = 1


прогон 2 | fetch_all  | 0.30 с | max_active = 10


прогон 3 | one_by_one | 3.01 с | max_active = 1


прогон 3 | fetch_all  | 0.30 с | max_active = 10


### Вывод

прогноз подтвердился

## Эксперимент 2. Один блокирующий вызов в event loop

Одна медленная задача (A: `time.sleep(1.0)` прямо в корутине; B: тот же sleep через `asyncio.to_thread`) + 5 быстрых `asyncio.sleep(0.1)` в одном `gather`.

### Прогноз

**1. В варианте A — на какой секунде закончатся 5 быстрых задач и почему?** – На 1, потому API несовместимы

**2. В варианте B — на какой секунде они закончатся, и сколько займёт весь `gather`?** – На 1, весь gather тоже займет 1 секунду

In [4]:
async def fast_task(t0: float) -> float:
    await asyncio.sleep(0.1)
    return round(time.perf_counter() - t0, 2)       # момент, когда быстрая задача закончилась

async def blocking_task() -> None:
    time.sleep(1.0)                                 # A: синхронный sleep внутри корутины

async def offloaded_task() -> None:
    await asyncio.to_thread(time.sleep, 1.0)        # B: тот же sleep, вынесенный в поток

for name, slow in [("A blocking", blocking_task), ("B to_thread", offloaded_task)]:
    t0 = time.perf_counter()
    res = await asyncio.gather(slow(), *[fast_task(t0) for _ in range(5)])
    print(name, "быстрые задачи закончились на:", res[1:], "| всего:", round(time.perf_counter() - t0, 2))

A blocking быстрые задачи закончились на: [1.11, 1.11, 1.11, 1.11, 1.11] | всего: 1.11


B to_thread быстрые задачи закончились на: [0.1, 0.1, 0.1, 0.1, 0.1] | всего: 1.0


### Вывод

*(записан Claude по просьбе Вадима — с разбором согласен)*

**A:** синхронный `time.sleep(1.0)` занял единственный поток event loop'а — быстрые задачи не стартовали, пока он не закончился, и завершились на 1.0 + 0.1 ≈ 1.1 с. Прогноз «на 1» верен по величине, но механизм — время блокировки **прибавляется** ко всем остальным задачам.

**B:** sleep вынесен в поток через `asyncio.to_thread`, loop свободен — быстрые задачи закончились на 0.1 с, весь `gather` ≈ max = 1.0 с. Ошибка в прогнозе B — неверно понял, о каких задачах вопрос.

### Разбор (от Claude, не мой вывод)

- Event loop — один поток. Корутина отдаёт управление только на `await` чего-то асинхронного; `time.sleep`, `requests.get`, тяжёлый CPU-цикл внутри `async def` держат поток, и все остальные задачи стоят.
- В A `gather` запустил медленную задачу первой → loop заблокирован на 1.0 с → быстрые задачи даже не начали свой `asyncio.sleep(0.1)` → закончились на 1.1. При delay быстрых 0.5 было бы 1.5.
- В B `asyncio.to_thread` отправляет `time.sleep` в пул потоков (`run_in_executor` с default `ThreadPoolExecutor`), а корутина ждёт future через `await` → loop свободен, быстрые задачи идут параллельно. `time.sleep` отпускает GIL, поэтому поток никому не мешает.
- Правило: блокирующий IO в async-коде → `asyncio.to_thread` (или async-библиотека: `aiohttp`, `httpx.AsyncClient`); CPU-bound → `loop.run_in_executor(ProcessPoolExecutor(), ...)` — в потоке он упрётся в GIL (см. 01_gil_demo, 03_multiprocessing).